# Predictive Circuit Coding Pipeline

One Colab runner for training, evaluation, crossed representation benchmarks, motif benchmarks, and optional diagnostics. The notebook is stage-resume oriented: rerunning the execution cell reuses completed stages when config and upstream artifacts still match.


In [ ]:
# Pipeline config
PIPELINE_CONFIG_PATH = "configs/pcc/pipeline_debug.yaml"  # switch to pipeline_full.yaml for the full claim-facing run
PIPELINE_RUN_ID = None  # None -> create a new run if training, else resume the latest compatible exported run


In [ ]:
# Setup: Drive mount, repo sync, install
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

repo_root = Path('/content/predictive_circuit_coding')
repo_url = os.environ.get('PCC_REPO_URL', 'https://github.com/jacobposchl/predictive_circuit_coding')
branch_name = os.environ.get('PCC_REPO_BRANCH')
if not repo_root.exists():
    clone_cmd = ['git', 'clone', repo_url, str(repo_root)]
    if branch_name:
        clone_cmd = ['git', 'clone', '--branch', branch_name, repo_url, str(repo_root)]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '%s[notebook]' % repo_root], check=True)
git_hash = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
print('Repo root:', repo_root)
print('Git commit:', git_hash)
os.chdir(repo_root)


In [ ]:
# Imports and preflight
from pathlib import Path
import pandas as pd

from predictive_circuit_coding.benchmarks import load_notebook_pipeline_config, run_notebook_pipeline_from_config
from predictive_circuit_coding.utils import build_notebook_preflight_rows, verify_paths_exist

pipeline_config = load_notebook_pipeline_config(repo_root / PIPELINE_CONFIG_PATH)
preflight_paths = {
    "pipeline_config": pipeline_config.config_path,
    "experiment_config": pipeline_config.experiment_config_path,
    "data_config": pipeline_config.data_config_path,
}
if pipeline_config.source_dataset_root is not None:
    preflight_paths["source_dataset_root"] = pipeline_config.source_dataset_root
status_df = pd.DataFrame(build_notebook_preflight_rows(path_status=verify_paths_exist(preflight_paths)))
display(status_df)
enabled_stages = [
    name
    for name, enabled in [
        ('train', pipeline_config.run_stage_train),
        ('evaluate', pipeline_config.run_stage_evaluate),
        ('representation_benchmark', pipeline_config.run_stage_representation_benchmark),
        ('motif_benchmark', pipeline_config.run_stage_motif_benchmark),
        ('alignment_diagnostic', pipeline_config.run_stage_alignment_diagnostic),
        ('image_identity_appendix', pipeline_config.run_stage_image_identity_appendix),
    ]
    if enabled
]
print('Pipeline config:', pipeline_config.config_path)
print('Run mode:', pipeline_config.notebook_ui.mode, '| log mode:', pipeline_config.notebook_ui.log_mode)
print('Enabled stages:', ', '.join(enabled_stages) if enabled_stages else 'none')
print('Local artifact root:', pipeline_config.local_artifact_root)
print('Drive export root:', pipeline_config.drive_export_root)
print('Source dataset root:', pipeline_config.source_dataset_root)
print('Stage prepared sessions locally:', pipeline_config.stage_prepared_sessions_locally)


In [ ]:
# Run or resume the pipeline
pipeline_result = run_notebook_pipeline_from_config(
    pipeline_config_path=repo_root / PIPELINE_CONFIG_PATH,
    pipeline_run_id=PIPELINE_RUN_ID,
)
print('Pipeline run ID:', pipeline_result.run_id)
print('Local run root:', pipeline_result.local_run_root)
print('Drive run root:', pipeline_result.drive_run_root)
print('Checkpoint:', pipeline_result.checkpoint_path)
print('Training summary:', pipeline_result.training_summary_path)
print('Representation summary:', pipeline_result.representation_summary_json_path)
print('Motif summary:', pipeline_result.motif_summary_json_path)
print('Final summary:', pipeline_result.final_summary_json_path)
print('Pipeline manifest:', pipeline_result.pipeline_manifest_path)
print('Pipeline state:', pipeline_result.pipeline_state_path)


In [ ]:
# Final visualization
from predictive_circuit_coding.utils import build_pipeline_summary_figure

alignment_df = pd.DataFrame()
if pipeline_result.alignment_summary_csv_path is not None and pipeline_result.alignment_summary_csv_path.is_file():
    alignment_df = pd.read_csv(pipeline_result.alignment_summary_csv_path)

summary_figure = build_pipeline_summary_figure(
    representation_df=representation_df,
    motif_df=motif_df,
    final_df=final_df,
    alignment_df=alignment_df,
)
display(summary_figure)


In [ ]:
# Inspect outputs
from predictive_circuit_coding.utils import load_pipeline_display_tables

tables = load_pipeline_display_tables(
    representation_summary_csv_path=pipeline_result.representation_summary_csv_path,
    motif_summary_csv_path=pipeline_result.motif_summary_csv_path,
    final_summary_csv_path=pipeline_result.final_summary_csv_path,
)
representation_df = tables['representation']
motif_df = tables['motif']
final_df = tables['final']

print('=== Representation Benchmark ===')
display(representation_df)
print('=== Motif Benchmark ===')
display(motif_df)
print('=== Final Project Summary ===')
display(final_df)

if pipeline_result.alignment_summary_json_path is not None and pipeline_result.alignment_summary_json_path.is_file():
    print('=== Alignment Diagnostic ===')
    display(pd.read_csv(pipeline_result.alignment_summary_csv_path))
